# 03 — Model Evaluation

**Project:** UREP 32-0210-250078 | Crack Classification

Test set evaluation (run once after training):
- Confusion matrix
- Classification report (precision, recall, F1)
- Misclassified examples
- Per-class accuracy breakdown

In [ ]:
import sys
sys.path.insert(0, "..")

import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

import config
from src.dataset import get_generators
from src.evaluation import evaluate_model

print(f"TensorFlow version: {tf.__version__}")

## Load Best Model

In [ ]:
model_path = os.path.join(config.MODEL_DIR, "best_model.keras")
print(f"Loading model from: {model_path}")
model = keras.models.load_model(model_path)
model.summary()

## Load Test Data

In [ ]:
_, _, test_gen = get_generators()
print(f"Test samples: {test_gen.samples}")
print(f"Test batches: {len(test_gen)}")
print(f"Class indices: {test_gen.class_indices}")

## Evaluate on Test Set

In [ ]:
metrics = evaluate_model(model, test_gen)

print("\n" + "=" * 40)
print("Summary Metrics:")
print(f"  Accuracy:          {metrics['accuracy']:.4f}")
print(f"  Precision (wt):    {metrics['precision_weighted']:.4f}")
print(f"  Recall (wt):       {metrics['recall_weighted']:.4f}")
print(f"  F1 Score (wt):     {metrics['f1_weighted']:.4f}")

## Per-Class Accuracy

In [ ]:
from sklearn.metrics import confusion_matrix

test_gen.reset()
y_pred_probs = model.predict(test_gen, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_gen.classes

cm = confusion_matrix(y_true, y_pred)
per_class_acc = cm.diagonal() / cm.sum(axis=1)

print("Per-Class Accuracy:")
for cls_name, acc in zip(config.CLASS_NAMES, per_class_acc):
    bar = "#" * int(acc * 40)
    print(f"  {cls_name:<12} {acc:.4f}  {bar}")

## Misclassified Examples

In [ ]:
from PIL import Image

# Find misclassified indices
misclassified_idx = np.where(y_pred != y_true)[0]
print(f"Total misclassified: {len(misclassified_idx)} / {len(y_true)}")

# Show up to 16 misclassified examples
n_show = min(16, len(misclassified_idx))
if n_show > 0:
    sample_idx = np.random.choice(misclassified_idx, size=n_show, replace=False)
    cols = 4
    rows = (n_show + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
    axes = axes.flat if n_show > 1 else [axes]

    for i, ax in enumerate(axes):
        if i < n_show:
            idx = sample_idx[i]
            img_path = test_gen.filepaths[idx]
            img = Image.open(img_path).convert("RGB")
            true_label = config.CLASS_NAMES[y_true[idx]]
            pred_label = config.CLASS_NAMES[y_pred[idx]]
            confidence = y_pred_probs[idx][y_pred[idx]]

            ax.imshow(img)
            ax.set_title(f"True: {true_label}\nPred: {pred_label} ({confidence:.1%})",
                        fontsize=9, color="red")
        ax.axis("off")

    plt.suptitle("Misclassified Examples", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("No misclassified examples — perfect score!")

## Confidence Distribution

In [ ]:
max_confidences = np.max(y_pred_probs, axis=1)
correct_mask = y_pred == y_true

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(max_confidences[correct_mask], bins=30, alpha=0.7, label="Correct", color="#2ecc71")
ax.hist(max_confidences[~correct_mask], bins=30, alpha=0.7, label="Incorrect", color="#e74c3c")
ax.axvline(x=config.CONFIDENCE_THRESHOLD, color="orange", linestyle="--",
           label=f"Threshold ({config.CONFIDENCE_THRESHOLD:.0%})")
ax.set_xlabel("Prediction Confidence")
ax.set_ylabel("Count")
ax.set_title("Confidence Distribution: Correct vs Incorrect Predictions")
ax.legend()
plt.tight_layout()
plt.show()

low_conf = np.sum(max_confidences < config.CONFIDENCE_THRESHOLD)
print(f"Predictions below {config.CONFIDENCE_THRESHOLD:.0%} confidence: {low_conf} / {len(max_confidences)}")